# 04 · MNIST 手写数字识别

这是深度学习的 **Hello World**：给一张 28×28 的手写数字图片，让模型认出它是 0~9 里的哪个。

这份笔记会新增 4 个概念：

| 新概念 | 一句话解释 |
|---|---|
| `DataLoader` | 自动把数据切成一批一批喂给模型 |
| 真实数据集 | 之前是自己造的数据，这次是 7 万张真图片 |
| `model.eval()` + `torch.no_grad()` | 测试时要关掉的东西 |
| 准确率 | 训练时看 loss，评价时看准确率 |

训练循环还是那熟悉的 5 行——**一行不多，一行不少**。

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# matplotlib 默认字体不含中文，图里的中文会显示成方块；指定微软雅黑解决
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False   # 让负号正常显示

print('torch', torch.__version__)

## 第 1 步 · 拿到数据

MNIST：70000 张 28×28 的手写数字灰度图，60000 张训练 + 10000 张测试。

`transforms.ToTensor()` 干两件事：

- 把像素值从 0~255 缩放到 **0~1**（神经网络喜欢小数字）
- 把图片变成形状 `(1, 28, 28)` 的张量（开头的 1 是颜色通道数，灰度图只有 1 个通道）

In [ ]:
transform = transforms.ToTensor()

# 路径写绝对路径，保证无论从哪里启动 Jupyter 都能找到
train_set = datasets.MNIST(root=r'D:\pytorch-notebooks\data', train=True,
                           download=True, transform=transform)
test_set  = datasets.MNIST(root=r'D:\pytorch-notebooks\data', train=False,
                           download=True, transform=transform)

print('训练集:', len(train_set), '张')
print('测试集:', len(test_set), '张')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
for i, ax in enumerate(axes.flat):
    img, label = train_set[i]          # 返回 (图片张量, 标签)
    ax.imshow(img[0], cmap='gray')     # img 形状是 (1,28,28)，取掉通道维
    ax.set_title(f'标签: {label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

对计算机来说，每张图只是一个 28×28 的数字表格。模型的任务：看这张表，输出 0~9。

## 第 2 步 · DataLoader：切成小批喂给模型

为什么不一次喂 6 万张？——又慢又占内存，而且学得更差。实践做法：切成 **batch**，一批 64 张。

`shuffle=True`：每轮开始前打乱顺序，防止模型"背顺序"而不是学规律。

之后你写训练代码时，只需要写 `for images, labels in train_loader:`，它就会一批一批吐数据。

In [ ]:
train_loader = DataLoader(train_set, batch_size=64,  shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

images, labels = next(iter(train_loader))   # 取出一批看看
print('一批图片的形状:', images.shape)       # (64, 1, 28, 28)
print('一批标签的形状:', labels.shape)       # (64,)
print('前 8 个标签:', labels[:8].tolist())

## 第 3 步 · 模型：最简单的全连接网络

```text
(1, 28, 28) --Flatten--> 784 个数 --Linear+ReLU--> 128 个数 --Linear--> 10 个数
```

- `nn.Flatten()` 把图片摊平成一维向量 784
- 最后输出 10 个数，**不是概率，是 10 个得分**——谁得分高就认成谁

这个结构叫 **MLP（多层感知机）**。它不看图片的空间结构，只是把 784 个像素点一股脑读进来——这正是它准确率的上限不高、而后面的 CNN 更强的原因。

In [ ]:
model = nn.Sequential(
    nn.Flatten(),            # (64, 1, 28, 28) -> (64, 784)
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'\n参数总数: {n_params:,}')   # 十万个参数，比 03 的两个参数多得多

## 第 4 步 · 损失函数 + 优化器

- `CrossEntropyLoss`：分类问题的标配。注意它**内部已经包含 softmax**，所以模型最后一层不要再加 softmax，直接输出得分就行——这是新手常踩的坑。
- 优化器用 SGD，学习率 0.1（分类任务常用的起点）。

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

## 第 5 步 · 训练——还是那 5 行

和 `03` 唯一的区别：数据不再是我们自己造的，而是 `for images, labels in train_loader:` 一批一批吐出来的。

**训练代码一行没变**。这就是 PyTorch 的好处：学会一次，到处能用。

跑完全部 60000 张图算 1 个 epoch，这里训 5 轮。CPU 上大约每轮十几秒，稍等。

In [ ]:
import time

EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    t0 = time.time()
    for images, labels in train_loader:
        pred = model(images)           # 1 前向传播
        loss = loss_fn(pred, labels)   # 2 计算损失
        optimizer.zero_grad()          # 3 梯度清零（忘这行=白训）
        loss.backward()                # 4 反向传播
        optimizer.step()               # 5 更新参数
        total_loss += loss.item()
    print(f'epoch {epoch+1}/{EPOCHS}  平均 loss={total_loss/len(train_loader):.4f}  用时 {time.time()-t0:.1f}s')

## 第 6 步 · 测试：准确率才是硬指标

两个新姿势：

- `model.eval()`：切换到"测试模式"。本例里影响不大，但这是标准写法，**现在就养成习惯**（用到 BatchNorm / Dropout 的模型不加它会算错）。
- `torch.no_grad()`：测试不需要求梯度，关掉更快、更省内存。

评价方式：拿模型从没见过的 10000 张测试图考它，答对的比例就是准确率。

In [ ]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        pred = model(images).argmax(1)   # 每张图取得分最高的那个数字
        correct += (pred == labels).sum().item()
        total += labels.size(0)

print(f'测试集准确率: {correct/total:.2%}  ({correct}/{total})')
print('参考线: 90% 说明训练有问题, 97% 是 MLP 正常水平, 98.5%+ 需要 CNN')

## 第 7 步 · 看看模型错在哪

只看一个准确率数字学不到东西。把认错的图挑出来看：**人一眼能认出来的，模型为什么认错？** 看懂这些错误，比多训 10 个 epoch 更有价值。

In [ ]:
wrong_imgs, wrong_pred, wrong_true = [], [], []
model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        pred = model(images).argmax(1)
        mask = pred != labels
        wrong_imgs.append(images[mask])
        wrong_pred.append(pred[mask])
        wrong_true.append(labels[mask])

wrong_imgs = torch.cat(wrong_imgs)
wrong_pred = torch.cat(wrong_pred)
wrong_true = torch.cat(wrong_true)
print(f'共认错 {len(wrong_imgs)} 张，展示前 8 张：')

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(wrong_imgs[i][0], cmap='gray')
    ax.set_title(f'认成 {wrong_pred[i].item()}，实际 {wrong_true[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 练习（做完才算过关）

1. **学习率实验**：把 `lr` 改成 `0.001` 和 `1.0` 各跑一次，观察 loss 差别。体会"步子太小走不动、步子太大到处蹦"。
2. **换优化器**：换成 `torch.optim.Adam(model.parameters(), lr=0.001)`，对比 SGD 的收敛速度。
3. **加深网络**：在 128 后面加 `nn.Linear(128, 64), nn.ReLU()`，看准确率变了多少。
4. **挑战（CNN 预告）**：把 Flatten 换成卷积结构试试：
   ```python
   nn.Conv2d(1, 8, 3), nn.ReLU(), nn.MaxPool2d(2)
   ```
   CNN 能把准确率推到 99%，全连接很难超过 98%——这就是第 4 阶段要学的东西。

## 复盘 + 三个坑

- **loss 一直不降**？九成是 lr 不合适，或者忘了 `optimizer.zero_grad()`。
- **报形状错误**？先打印 `images.shape`，回到第 2 步核对每一维是什么。
- **CrossEntropyLoss 的输入是原始得分**，最后一层不要加 softmax。